# 14. Pestaña «EDA»: interpretación de cada resultado

La pestaña de EDA del dashboard resume los hallazgos del Entregable 1 que guían el modelado.
Todas sus figuras usan **solo el entrenamiento**. Para evitar una página excesivamente larga se organiza en cuatro **subpestañas internas**
(la pestaña principal sigue siendo una sola):

| Subpestaña | Contenido | Sección |
|---|---|---|
| A · Objetivo y calidad | tarjetas de calidad de datos + origen del objetivo | 14.1 |
| B · Relaciones con el objetivo | tasa por categoría, hora × modo, numéricas | 14.2–14.4 |
| C · Tiempo y espacio | serie semanal por mercado, mapa | 14.5–14.6 |
| D · Auditoría de fuga | disponibles vs posteriores al pedido, con/sin fuga | 14.7 |

Aquí se interpreta cada figura y se añaden las pruebas estadísticas y tablas que el dashboard no muestra.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

import plotly.io as pio
from IPython.display import HTML
sys.path.insert(0, str(project_root / "dashboard"))
import figuras as F
import textos as T
_plotly = {"cargado": False}

def mostrar(fig):
    """Muestra una figura Plotly del dashboard (la primera de la página carga plotly.js desde la CDN)."""
    incluir = "cdn" if not _plotly["cargado"] else False
    _plotly["cargado"] = True
    display(HTML(pio.to_html(fig, full_html=False, include_plotlyjs=incluir, config={"displaylogo": False, "responsive": True})))

from scipy import stats
TAB = project_root / "tables"

## 14.1 Objetivo y calidad de datos

### Resumen de calidad (tarjetas)

Las tarjetas **no calculan nada nuevo**: resumen resultados ya obtenidos en los capítulos 2 y 9 (`faltantes_por_columna.csv`, `auditoria_duplicados.csv`,
`valores_inconsistentes.csv`, `auditoria_fuga.csv`); el único dato recalculado por el código es el de **pedidos compartidos entre entrenamiento y prueba**, que
se vuelve a comprobar con los datos en cada ejecución de `preparar_datos.py`. El detalle completo queda en el libro; el dashboard muestra solo el resumen.

In [2]:
q = F.calidad()
f, i, inc, fu, v = q["faltantes"], q["integridad"], q["inconsistencias"], q["fuga"], q["variables"]
tarjetas = pd.DataFrame([
    ("Variables con faltantes", f"{f['n_variables']} de {v['originales']}", ", ".join(f"{k} ({F.pct(x / 100)})" for k, x in f["detalle"].items())),
    ("Mayor porcentaje de faltantes", F.pct(f["pct_max"] / 100, 0), f["variable_max"]),
    ("Columnas duplicadas o equivalentes", i["columnas_duplicadas"], f"{i['lineas_compartidas_train_test']} líneas (order_item_id) repetidas entre conjuntos"),
    ("Pedidos compartidos TRAIN/TEST", i["pedidos_compartidos_train_test"], f"días entre el último pedido de train y el primero de test: {i['dias_entre_train_y_test']:.0f}"),
    ("Comprobaciones de consistencia con hallazgos", f"{inc['con_hallazgos']} de {inc['comprobaciones']}", "; ".join(f"{k}: {es(x, 0)}" for k, x in inc["detalle"].items())),
    ("Variables de alto riesgo de fuga", len(fu["alto_riesgo"]), ", ".join(fu["alto_riesgo"])),
], columns=["tarjeta", "valor", "detalle"])
display(tarjetas)
md_out(f"**Pedidos compartidos entre entrenamiento y prueba = {i['pedidos_compartidos_train_test']}** (también 0 entre entrenamiento y gap). "
       f"Los {es(i['clientes_en_ambos_conjuntos'], 0)} clientes que aparecen en ambos conjuntos son esperables y no se usan como predictor. "
       f"Con una partición aleatoria por filas, el {es(i['pct_validacion_con_pedido_en_train_si_fuera_aleatoria'], 1)} % de las líneas de validación tendría su pedido en el entrenamiento: "
       "por eso la partición es cronológica y por pedido completo.")

,tarjeta,valor,detalle
0,Variables con faltantes,2 de 53,"product_description (100,0 %), order_zipcode (82,6 %)"
1,Mayor porcentaje de faltantes,100 %,product_description
2,Columnas duplicadas o equivalentes,8,0 líneas (order_item_id) repetidas entre conjuntos
3,Pedidos compartidos TRAIN/TEST,0,días entre el último pedido de train y el primero de test: 7
4,Comprobaciones de consistencia con hallazgos,3 de 13,Tardía según días reales > programados pero etiqueta 0: 3.362; pro...
5,Variables de alto riesgo de fuga,4,"days_shipping_real, delivery_status, order_status, shipping_date"


**Pedidos compartidos entre entrenamiento y prueba = 0** (también 0 entre entrenamiento y gap). Los 6.426 clientes que aparecen en ambos conjuntos son esperables y no se usan como predictor. Con una partición aleatoria por filas, el 89,6 % de las líneas de validación tendría su pedido en el entrenamiento: por eso la partición es cronológica y por pedido completo.

**Interpretación.**

- **Faltantes:** solo dos de las 53 variables originales tienen valores faltantes y ninguna es candidata a predictora: `product_description` está vacía (100 %) y
  `order_zipcode` falta en ≈ 83 % de las líneas (la asociación con otras variables está en `asociacion_faltantes.csv`). Las candidatas no tienen faltantes; el *Pipeline* igualmente imputa por robustez.
- **Integridad:** ocho columnas son copia exacta o equivalente de otra (capítulo 2) y se conserva una por par. No hay líneas ni pedidos repetidos entre conjuntos.
- **Consistencia:** de 13 comprobaciones, solo 3 tienen casos. La relevante es la de las líneas «tardías por días reales» con etiqueta 0, que son los cancelados/fraude; las otras dos
  indican columnas constantes o enmascaradas (`product_status`, correo y contraseña del cliente). Ninguna comprobación aritmética o de rango falla.
- **Fuga:** cuatro variables tienen riesgo alto (días reales, estado de la entrega, estado del pedido y fecha de despacho); son posteriores al pedido y se excluyen (14.7). Los identificadores `order_id` y `order_item_id`
  tienen riesgo medio porque codifican el tiempo y solo se usan para agrupar.

### Origen del objetivo

In [3]:
mostrar(F.fig_objetivo())
d = F.tabla("objetivo_por_estado.csv").pivot(index="delivery_status", columns="late_delivery_risk", values="lineas").fillna(0).astype(int)
d.columns = ["Late_delivery_risk = 0", "Late_delivery_risk = 1"]
d

,Late_delivery_risk = 0,Late_delivery_risk = 1
delivery_status,,
Advance shipping,32954,0
Late delivery,0,78549
Shipping canceled,6032,0
Shipping on time,25540,0


**Interpretación.** La etiqueta es una recodificación del estado de entrega: «Late delivery» es
siempre 1 y los otros tres estados son siempre 0. El estado **«Shipping canceled»** (≈ 4 % de las
líneas) agrupa pedidos cancelados y sospechosos de fraude que **nunca se entregaron**: su 0 **no
significa «llegó a tiempo»**, por eso en la figura aparece en gris, separado de las entregas adelantadas o a tiempo. Se excluyen de la población de modelado (capítulo 9) y, por
la misma razón, `delivery_status` no puede usarse como predictor: es la respuesta.

**No aparece en el dashboard:** la razón de desbalance es 1,22 (54,9 % vs 45,1 %) y la proporción
es la misma a nivel de pedido (54,8 %), así que no hace falta reponderar clases.

**Detalles que no aparecen en el dashboard.** El dashboard solo muestra las tarjetas de resumen. La tabla de arriba da el detalle de cada tarjeta y los archivos de origen
(`faltantes_por_columna.csv`, `asociacion_faltantes.csv`, `auditoria_duplicados.csv`, `valores_inconsistentes.csv`, `auditoria_fuga.csv`) están en los capítulos 2 y 9.

## 14.2 Tasa de entregas tardías por variable categórica (control interactivo)

En el dashboard un menú permite elegir la variable. Aquí se muestran las más relevantes.

In [4]:
for v in ["shipping_mode", "market", "payment_type"]:
    mostrar(F.fig_tasa_categoria(v))

In [5]:
cram = pd.read_csv(TAB / "cramer_pares.csv")
con_obj = cram[(cram["var_2"] == "late_delivery_risk") | (cram["var_1"] == "late_delivery_risk")].copy()
con_obj["variable"] = np.where(con_obj["var_1"] == "late_delivery_risk", con_obj["var_2"], con_obj["var_1"])
display(con_obj[["variable", "V_cramer", "chi2", "p_fdr"]].sort_values("V_cramer", ascending=False).round(4).head(10))

,variable,V_cramer,chi2,p_fdr
11,shipping_mode,0.4584,30069.4670,0.0000
22,payment_type,0.0754,816.4063,0.0000
49,order_country,0.0547,591.0679,0.0000
67,customer_state,0.0239,124.7233,0.0000
41,order_region,0.0175,65.8869,0.0000
32,market,0.0040,6.2814,0.3058
77,order_weekend,0.0024,1.7948,0.3058
76,product_name,0.0012,53.1929,0.6300
62,customer_country,0.0000,0.2820,0.7036
56,customer_segment,0.0000,0.0275,0.9863


**Interpretación.**

- **Modo de envío:** es la única categórica con asociación relevante (V de Cramér ≈ 0,46).
  First Class se retrasa ≈ 95 % de las veces, Second Class ≈ 77 %, Same Day ≈ 45 % y Standard
  ≈ 38 %. Paradójicamente, **los envíos más rápidos prometidos son los que más fallan**, porque su
  promesa (1 o 2 días) es la más ajustada.
- **Mercado de destino:** todas las barras quedan dentro de ± 1 punto de la tasa global, con
  intervalos de confianza que se solapan: no se observan diferencias relevantes entre mercados.
- **Medio de pago:** TRANSFER aparece algo por debajo porque concentra los pedidos sospechosos de
  fraude (etiqueta 0). Al excluirlos, la diferencia desaparece (AUC 0,532 → 0,504, capítulo 9).

**No aparece en el dashboard:** con n ≈ 143.000 casi todas las pruebas χ² resultan significativas
tras la corrección FDR; por eso se reporta el **tamaño de efecto** (V de Cramér), que es
despreciable (< 0,1) para todo salvo el modo de envío.

## 14.3 Hora del pedido × modo de envío

In [6]:
mostrar(F.fig_hora_modo())
display(pd.read_csv(TAB / "tasa_modo_jornada.csv", index_col=0).round(3))

,mean_AM,mean_PM,size_AM,size_PM
shipping_mode,,,,
First Class,0.956,0.955,10517,11501
Same Day,0.000,0.959,4110,3602
Second Class,0.769,0.766,14091,13874
Standard Class,0.380,0.383,42781,42599


**Interpretación.** Es el hallazgo más útil del EDA. En **Same Day**, los pedidos hechos antes de
las 12:00 casi nunca llegan tarde (0 %) y los de la tarde casi siempre (≈ 96 %): hay una **hora de
corte** operativa. En First Class todas las horas están en ≈ 95 % y en Standard y Second Class la
hora no influye. Como el efecto de la hora existe en un solo modo, un modelo aditivo (modo + hora)
no puede representarlo; por eso el modelo base usa la **interacción modo × jornada (AM/PM)**.

**No aparece en el dashboard:** el capítulo 9 muestra el mecanismo con las variables posteriores
(usadas solo para entender, no para predecir): Standard y Second Class tienen días reales uniformes
entre 2 y 6, First Class tarda siempre 2 días y Same Day tarda 0 o 1 día según la jornada. Por eso
el retraso en Standard y Second Class parece **aleatorio** con la información disponible al pedido: no se encontró ninguna variable que lo explique.

## 14.4 Variables numéricas según el objetivo (control interactivo)

In [7]:
for v in ["sales", "order_item_profit_ratio", "days_shipping_scheduled"]:
    mostrar(F.fig_numerica(v))
mw = pd.read_csv(TAB / "numericas_vs_objetivo.csv")
mw[["variable", "mediana_tardía", "mediana_no_tardía", "rank_biserial", "p_fdr", "efecto"]].sort_values("rank_biserial", key=abs, ascending=False).round(4)

,variable,mediana_tardía,mediana_no_tardía,rank_biserial,p_fdr,efecto
8,days_shipping_scheduled,2.0000,4.0000,-0.3955,0.0000,mediano
11,order_hour,12.0000,11.0000,0.0561,0.0000,despreciable
9,items_in_order,4.0000,4.0000,0.0060,0.2289,despreciable
6,benefit_per_order,30.8900,31.2900,-0.0038,0.7212,despreciable
13,order_month,6.0000,6.0000,0.0037,0.7212,despreciable
1,order_item_product_price,59.9900,59.9900,-0.0028,0.8458,despreciable
0,sales,179.9700,179.9700,-0.0021,0.8458,despreciable
5,order_item_profit_ratio,0.2700,0.2700,-0.0019,0.8458,despreciable
7,sales_per_customer,159.9900,159.9900,-0.0019,0.8458,despreciable
12,order_dow,3.0000,3.0000,0.0018,0.8458,despreciable


**Interpretación.** Las distribuciones de ventas o margen son prácticamente idénticas para
líneas tardías y puntuales: las barras roja y azul se superponen. La única numérica con efecto
apreciable es `days_shipping_scheduled` (rank-biserial ≈ −0,40), que es el modo de envío expresado
en días. **No aparece en el dashboard:** la prueba de Mann-Whitney con corrección FDR y el tamaño de
efecto de cada variable (tabla de arriba); y que el VIF de ventas, total y descuento es enorme
porque son combinaciones aritméticas (capítulo 5).

## 14.5 Serie semanal por mercado (control interactivo)

In [8]:
mostrar(F.fig_serie("Todos"))
mostrar(F.fig_serie("Europe"))
display(F.tabla("estacionariedad.csv").round(4))
display(F.tabla("ljung_box_tasa.csv").rename(columns={"Unnamed: 0": "rezago (días)"}).round(4))
display(F.tabla("mercado_por_trimestre.csv"))

,serie,ADF estadístico,ADF p,KPSS estadístico,KPSS p,conclusión
0,tasa diaria de tardías,-21.5872,0.0,0.0466,0.1,estacionaria
1,líneas por día,-28.7777,0.0,0.0567,0.1,estacionaria


,rezago (días),lb_stat,lb_pvalue
0,7,5.2689,0.6272
1,14,9.4708,0.7998
2,28,22.3339,0.7656


,order_date,Africa,Europe,LATAM,Pacific Asia,USCA
0,2015Q1,0,0,15413,0,0
1,2015Q2,0,5255,10362,0,0
2,2015Q3,0,15712,0,0,0
3,2015Q4,0,3947,0,11961,0
4,2016Q1,0,0,0,15421,0
5,2016Q2,0,0,0,159,15294
6,2016Q3,3071,983,0,1959,9786
7,2016Q4,7285,2889,0,5102,601
8,2017Q1,1258,563,12493,1038,118
9,2017Q2,0,0,2405,0,0


**Interpretación.** La tasa semanal oscila alrededor del 55 %. **En los análisis realizados no se observa una tendencia o estacionalidad relevante para
explicar el objetivo.** Sobre la serie **diaria** (capítulo 7), ADF rechaza raíz unitaria y KPSS no rechaza estacionariedad: la serie es
compatible con una serie **estacionaria**. Ljung-Box no detecta autocorrelación hasta 28 días (la tasa de un día no anticipa
las siguientes), por lo que no se usan rezagos del objetivo como predictores.

Al elegir un mercado, la serie aparece **solo en algunos tramos**: los mercados de destino **rotan
en el tiempo** (tabla de líneas por trimestre: 2015-T1 solo LATAM, 2015-T3 solo Europa, 2016-T1
solo Pacific Asia...). Es un artefacto de la simulación que confunde destino con periodo; por eso
el destino no entra al modelo (capítulo 10).

In [9]:
est = F.tabla("estacionariedad.csv").set_index("serie")
lb = F.tabla("ljung_box_tasa.csv").rename(columns={"Unnamed: 0": "rezago"}).set_index("rezago")
s = est.loc["tasa diaria de tardías"]
md_out(f"""**Detalles que no aparecen en el dashboard.** El dashboard solo grafica la serie **semanal**; las pruebas se hicieron sobre la serie **diaria**: para la tasa diaria de tardías, ADF = {F.num(s['ADF estadístico'], 2)} (p < 0,001) y
KPSS = {F.num(s['KPSS estadístico'])} (p = {F.num(s['KPSS p'], 1)}, el límite superior que reporta la prueba) → «{s['conclusión']}». Ljung-Box no detecta autocorrelación: p = {F.num(lb.loc[7, 'lb_pvalue'], 2)} a 7 días,
{F.num(lb.loc[14, 'lb_pvalue'], 2)} a 14 y {F.num(lb.loc[28, 'lb_pvalue'], 2)} a 28. Las tablas completas (estacionariedad, Ljung-Box y líneas por mercado y trimestre) están arriba.""")

**Detalles que no aparecen en el dashboard.** El dashboard solo grafica la serie **semanal**; las pruebas se hicieron sobre la serie **diaria**: para la tasa diaria de tardías, ADF = -21,59 (p < 0,001) y
KPSS = 0,047 (p = 0,1, el límite superior que reporta la prueba) → «estacionaria». Ljung-Box no detecta autocorrelación: p = 0,63 a 7 días,
0,80 a 14 y 0,77 a 28. Las tablas completas (estacionariedad, Ljung-Box y líneas por mercado y trimestre) están arriba.

## 14.6 Mapa de la tasa por celda

In [10]:
mostrar(F.fig_mapa())
display(F.tabla("moran_global.csv").round(4))
display(pd.read_csv(TAB / "maup_moran.csv").round(4))
display(pd.read_csv(TAB / "knox.csv").round(3))

,variable,I,E[I],p_permutacion,z
0,tasa de tardías,0.0153,-0.005,0.570,0.5267
1,% First Class,0.0027,-0.005,0.840,0.2256
2,log(líneas) (densidad),0.1097,-0.005,0.002,3.6354


,celda_grados,celdas,I_Moran,p
0,0.5,185,0.0003,0.888
1,1.0,203,0.0153,0.558
2,2.0,116,-0.0148,0.872
3,4.0,52,-0.0664,0.388


,pares_cercanos_espacio,pares_cercanos_espacio_y_tiempo,esperado_bajo_independencia,p_permutacion
0,152056,2417,2528.925,0.995


**Interpretación.** Las celdas rojas y azules se mezclan sin formar zonas visibles. **No se detectó evidencia estadísticamente significativa de autocorrelación espacial con la prueba utilizada**:
la I de Moran global de la tasa es ≈ 0 y no significativa con los tamaños de celda evaluados (sin efecto MAUP aparente), y la prueba de Knox no
detecta agrupamiento conjunto en espacio y tiempo. Las celdas más extremas son las pequeñas
(puntos chicos), donde el azar pesa más. La única estructura espacial detectada es la **densidad** de
clientes (I de Moran significativa para el número de líneas): hay clientes en ciudades, no
retrasos en ciudades.

**No aparece en el dashboard:** las coordenadas corresponden al cliente/tienda, no al destino; 17
ubicaciones tienen errores de georreferenciación (longitud con signo invertido) y se excluyeron
del mapa; el mapa omite Hawái por escala. Una prueba no significativa no demuestra ausencia de patrón: solo indica que, con esta prueba y esta resolución, no hay evidencia de él.

## 14.7 Auditoría de fuga

La subpestaña separa explícitamente dos grupos de variables, a partir de `auditoria_fuga.csv` (capítulo 9):

- **Disponibles al crear el pedido:** se conocen cuando el cliente compra; solo ellas pueden ser predictoras.
- **Posteriores al pedido (fuga):** `days_shipping_real`, `delivery_status`, `order_status` y `shipping_date`. Se resaltan en rojo y **no se usan en ningún modelo presentado como válido**;
  solo aparecen en la comparación con/sin fuga como demostración metodológica.

In [11]:
fu = F.calidad()["fuga"]
aud = pd.read_csv(TABLES / "auditoria_fuga.csv")
disp = aud[aud["variable"].isin(fu["disponibles_al_pedido"])][["variable", "disponible_al_predecir", "AUC univariado"]]
post = aud[aud["variable"].isin(fu["posteriores_al_pedido"])][["variable", "disponible_al_predecir", "evidencia", "AUC univariado"]]
md_out("**Disponibles al crear el pedido (candidatas):** " + ", ".join(f"`{v}`" for v in fu["disponibles_al_pedido"]))
md_out("**Posteriores al pedido (fuga):**")
display(post.round(4))
mostrar(F.fig_fuga())
mostrar(F.fig_con_sin_fuga())

**Disponibles al crear el pedido (candidatas):** `payment_type`, `category_name`, `customer_segment`, `department_name`, `latitude`, `longitude`, `market`, `order_country`, `order_item_discount_rate`, `order_item_profit_ratio`, `order_item_quantity`, `sales`, `order_region`, `shipping_mode`, `order_weekend`, `order_afternoon`, `mode_x_daypart`, `items_in_order`, `orders_prev_7d`

**Posteriores al pedido (fuga):**

,variable,disponible_al_predecir,evidencia,AUC univariado
1,days_shipping_real,No,Se conoce cuando el pedido se envía/entrega; el objetivo es días r...,0.7098
5,delivery_status,No,Estado final de la entrega; «Late delivery» ES la etiqueta 1.,1.0000
42,order_status,No,"Estado del pedido en el sistema (COMPLETE, CLOSED, CANCELED, SUSPE...",0.5415
51,shipping_date,No,Fecha de despacho = fecha del pedido + días reales; posterior y eq...,NaN


**Interpretación.** En rojo, las variables que solo se conocen **después** de crear el pedido (y las derivadas de ellas, como la diferencia días reales − programados).
`delivery_status` separa perfectamente (AUC = 1) y la diferencia días reales − programados casi
también (≈ 0,97). Un modelo que las use alcanza ROC-AUC ≈ 0,975: no predice, **lee la respuesta**.
Sin ellas, la **referencia empírica** basada en modo × jornada alcanza ≈ 0,775: es un **benchmark observado con la información disponible**
(no un máximo demostrado). Este contraste es la razón por la que muchos trabajos publicados sobre DataCo
reportan resultados casi perfectos, y por la que este proyecto trabaja con la versión sin fuga.

**No aparece en el dashboard:** `days_shipping_real` sola tiene un AUC de apenas 0,71 (parecido al
del modo de envío) y no dispararía una alerta univariada; la fuga aparece al combinarla con los días
programados. Por eso la auditoría se basa en la **disponibilidad temporal** de cada variable, no
solo en su AUC. `shipping_date` es una fecha-hora y no tiene AUC univariado en esta tabla; se excluye por su disponibilidad (despacho posterior al pedido).
La barra de la referencia en la comparación con/sin fuga se calcula sobre el entrenamiento (capítulo 9); su equivalente en la prueba (capítulo 15) es prácticamente igual.